In [1]:
import numpy as np
import commpy as cp
# import matplotlib.pyplot as plt

In [2]:
M = 16
SNR = 20
D_noise = 10**(-SNR/10)

modem = cp.QAMModem(M)

input_bits = np.random.randint(0, 2, size=10000)

Модулированные символы:
$$
I= \{I_i \}_{i=0}^{N-1};
$$

In [3]:
input_signal = modem.modulate(input_bits) / np.sqrt(modem.Es)
N = len(input_signal)

Импульсная характеристика многолучевого канала:
$$
h=\{h_i \}_{i=0}^{L-1};
$$

In [26]:
L = 2
h = 10**(-np.linspace(0, 10, L) / 10)

Сигнал на выходе из канала:
$$\nu = h * I + n \quad \Longleftrightarrow \quad \nu_k = \sum\limits_{i=0}^{L-1}h_i \cdot I_{k-i} + n_k. \quad (k - i \geq 0)$$

In [27]:
output_signal = np.zeros_like(input_signal)
for k in range(N):
    for i in range(L):
        # Свёртка с ИХ канала
        if k-i >= 0:
            output_signal[k] += h[i] * input_signal[k-i]
    # Добавление шума
    output_signal[k] += (np.random.randn() + 1j * np.random.randn()) * np.sqrt(D_noise / 2)

Запускаем сигнал на фильтр прямой связи (**FeedForward Filter**):
$$
y = w^{ff} * \nu \quad \Longleftrightarrow \quad y_k = \sum\limits_{i=0}^{K_1-1}w^{ff}_i \cdot \nu_{k-i}, \quad (k - i \geq 0)
$$
где $w^{ff}$ - веса фильтра прямой связи.
Для определения фильтра с обратной связью по решению, рассмотрим поподробнее сигнал на выходе из фильтра прямой связи:
$$
\begin{aligned}
    y_k = \sum\limits_{i=0}^{K_1-1}w^{ff}_i \cdot \nu_{k-i} = \sum\limits_{i=0}^{K_1-1}w^{ff}_i \cdot (\sum\limits_{l=0}^{L-1}h_l \cdot I_{k-i-l} + n_{k-i})&= w^{ff}_0h_0I_k + w^{ff}_0h_1I_{k-1} + w^{ff}_0h_2I_{k-2} + \ldots + \\
    &+w^{ff}_1h_0I_{k-1} + w^{ff}_1h_1I_{k-2} + w^{ff}_1h_2I_{k-3} + \ldots + \\
    &+w^{ff}_2h_0I_{k-2} + w^{ff}_2h_1I_{k-3} + w^{ff}_2h_2I_{k-4} + \ldots + \\
    &+ \ldots + \\
    &+\sum\limits_{i=0}^{K_1-1}w^{ff}_i \cdot n_{k-i} = \\
    &= w^{ff}_0h_0I_k + (w^{ff}_0h_1 + w^{ff}_1h_0)\cdot I_{k-1} + \\
    &+ (w^{ff}_0h_2 + w^{ff}_1h_1 + w^{ff}_2h_0) \cdot I_{k-2} + \\
    &+ (w^{ff}_0h_3 + w^{ff}_1h_2 + w^{ff}_2h_1 + w^{ff}_3h_0) \cdot I_{k-3} + \\
    &+\ldots+ \\
    &+\sum\limits_{i=0}^{K_1-1}w^{ff}_i \cdot n_{k-i} = \\
    &=w^{ff}_0h_0I_k + \sum_{i=1}^{K_2-1}I_{k-i}\sum_{l=0}^i w^{ff}_lh_{i-l} + \sum_{i=0}^{K_1-1}w^{ff}_i \cdot n_{k-i},
\end{aligned}
$$
где $K_2 = 2(K_1 - 1) + 1$.

Получаем сумму основного сигнала и прекурсоров. Для того чтобы избавиться от них, пропустим сигнал через фильтр с обратной связью по решению (**FeedBack Filter**):
$$
\hat{I} = y - w^{bf} * \hat{I}' \quad \Longleftrightarrow \quad \hat{I}_k = y_k - \sum_{i=1}^{K_2-1}w^{bf}_i \cdot \hat{I}_{k-i}'.
$$
Тогда эквализированный сигнал будет выглядеть следующим образом:
$$
\hat{I}_k = w^{ff}_0h_0I_k + \sum_{i=1}^{K_2-1}\left(I_{k-i}\sum_{l=0}^i w^{ff}_lh_{i-l} - \hat{I}_{k-i} \cdot w^{bf}_i \right) + \sum_{i=0}^{K_1-1}w^{ff}_i \cdot n_{k-i} \quad \Rightarrow \quad w^{bf}_i = \sum_{l=0}^i w^{ff}_lh_{i-l}.
$$
Теперь определим веса фильтра прямой связи по критерию пикового искажения (Zero-Forcing):
$$
w^{ff} = \{ w^{ff}_i \}_{i=0}^{K_1-1} = \{ 1 / h_i \}_{i=0}^{L-1}.
$$
Формулы эквализированного сигнала и весов фильтра обратной связи по решению будут выглядеть следующий образом:
$$
\hat{I}_k = I_k + \sum_{i=1}^{2L-2}\left(I_{k-i}\sum_{l=0}^i h_{i-l}/h_l - \hat{I}_{k-i} \cdot w^{bf}_i \right) + \sum_{i=0}^{L-1}n_{k-i}/h_i; \quad w^{bf} = \{ w^{bf}_i \}_{i=1}^{2L-2} = \{ \sum_{l=0}^i h_{i-l}/h_l \}_{i=1}^{2L-2}.
$$

In [29]:
# Количество ячеек памяти FeedForward Filter
K1 = len(h)
# Веса FeedForward Filter по критерию Zero-Forcing
w_ff = (h**(-1))

# Количество ячеек памяти FeedBack Filter
K2 = 2 * (K1 - 1) + 1
# Веса FeedBack Filter
w_bf = np.zeros(K2)
for i in range(1, K2):
    for l in range(K1):
        if 0 <= i-l < len(h):
            w_bf[i] += w_ff[l] * h[i-l]
        else:
            continue

# Сигнал после прямого фильтра
y = np.zeros_like(output_signal)
for k in range(N):
    for i in range(K1):
        if k-i >= 0:
            y[k] += w_ff[i] * output_signal[k-i]
        else:
            break

for k in range(len(output_signal)):
    for i in range(1, K2):
        if k - i >= 0:
            y[k] -= w_bf[i] * input_signal[k-i]
        else:
            break

output_bits = modem.demodulate(y*np.sqrt(modem.Es), demod_type='hard')

ber = np.mean((input_bits + output_bits) % 2)

print(ber)

0.2904
